In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass

import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

SEED = 42
TRAIN_PATH = 'train_clean.csv'
SUB_PATH = 'sample_submission.csv'
HORIZON = 56
VAL_DAYS = 28

np.random.seed(SEED)
print('HAS_XGB =', HAS_XGB)


HAS_XGB = True


In [2]:
print('[1] Loading train_clean.csv...')
float_cols = ['Quantity', 'UnitPrice', 'SalesAmount', 'Unit Cost', 'Cost Amount']
usecols = ['Date', 'ItemCode'] + float_cols

df_raw = pd.read_csv(
    TRAIN_PATH,
    usecols=usecols,
    dtype={'ItemCode': 'category'},
    parse_dates=['Date']
)

for col in float_cols:
    df_raw[col] = (df_raw[col].astype(str)
                   .str.replace(',', '.', regex=False)
                   .astype('float32'))

print(f"Raw: {df_raw.shape}, dates {df_raw['Date'].min().date()} -> {df_raw['Date'].max().date()}")
print(df_raw.dtypes)

df_raw.head(3)


[1] Loading train_clean.csv...
Raw: (711980, 7), dates 2020-11-17 -> 2025-09-05
Date           datetime64[us]
ItemCode             category
Quantity              float32
UnitPrice             float32
SalesAmount           float32
Unit Cost             float32
Cost Amount           float32
dtype: object


,Date,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,SKU-08063,12.0,242700.0000,2184300.0,123559.101562,1482709.0
1,2020-11-17,SKU-09458,600.0,131818.1875,79090912.0,110000.000000,66000000.0
2,2020-11-18,SKU-08062,6.0,230000.0000,940909.0,101000.000000,606000.0


In [3]:
print('[2] Aggregate to daily SKU and handle returns...')

daily = (
    df_raw.groupby(['Date', 'ItemCode'], observed=True, as_index=False)
    .agg({
        'Quantity': 'sum',
        'SalesAmount': 'sum',
        'Cost Amount': 'sum',
        'UnitPrice': 'mean',
        'Unit Cost': 'mean'
    })
    .rename(columns={'Cost Amount': 'CostAmount'})
)

# Strategy: clip net daily quantity to 0
# (returns reduce same-day demand but target remains non-negative for training)
daily['y'] = daily['Quantity'].clip(lower=0).astype('float32')
daily['profit'] = (daily['SalesAmount'] - daily['CostAmount']).astype('float32')

print('daily shape:', daily.shape)
print('negative y rows:', int((daily['y'] < 0).sum()))
daily.head(3)


[2] Aggregate to daily SKU and handle returns...
daily shape: (507050, 9)
negative y rows: 0


,Date,ItemCode,Quantity,SalesAmount,CostAmount,UnitPrice,Unit Cost,y,profit
0,2020-11-17,SKU-08063,12.0,2184300.0,1482709.0,242700.0000,123559.101562,12.0,701591.0
1,2020-11-17,SKU-09458,600.0,79090912.0,66000000.0,131818.1875,110000.000000,600.0,13090912.0
2,2020-11-18,SKU-08062,6.0,940909.0,606000.0,230000.0000,101000.000000,6.0,334909.0


In [4]:
print('[3] Build full panel...')
all_dates = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
all_skus = daily['ItemCode'].cat.categories if str(daily['ItemCode'].dtype) == 'category' else pd.Index(daily['ItemCode'].unique())

panel = pd.MultiIndex.from_product([all_dates, all_skus], names=['Date', 'ItemCode']).to_frame(index=False)
panel['ItemCode'] = panel['ItemCode'].astype('category')

panel = panel.merge(daily, on=['Date', 'ItemCode'], how='left')
for c in ['Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'y', 'profit']:
    panel[c] = panel[c].fillna(0).astype('float32')

print('panel shape:', panel.shape)
print('n_skus:', panel['ItemCode'].nunique(), 'n_days:', panel['Date'].nunique())
panel.head(3)


[3] Build full panel...
panel shape: (28014888, 9)
n_skus: 15972 n_days: 1754


,Date,ItemCode,Quantity,SalesAmount,CostAmount,UnitPrice,Unit Cost,y,profit
0,2020-11-17,SKU-00002,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2020-11-17,SKU-00003,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2020-11-17,SKU-00005,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['dow'] = out['Date'].dt.dayofweek.astype('int8')
    out['dom'] = out['Date'].dt.day.astype('int8')
    out['month'] = out['Date'].dt.month.astype('int8')
    out['quarter'] = out['Date'].dt.quarter.astype('int8')
    out['woy'] = out['Date'].dt.isocalendar().week.astype('int16')
    out['is_weekend'] = (out['dow'] >= 5).astype('int8')
    out['is_month_start'] = out['Date'].dt.is_month_start.astype('int8')
    out['is_month_end'] = out['Date'].dt.is_month_end.astype('int8')
    return out


def add_lag_rolling_features(df: pd.DataFrame, lags=(1,7,14,28,56), rolls=(7,14,28,56)) -> pd.DataFrame:
    out = df.copy()
    g = out.groupby('ItemCode', observed=True, sort=False)['y']

    for lag in lags:
        out[f'lag_{lag}'] = g.shift(lag).astype('float32')

    s1 = g.shift(1)
    for w in rolls:
        out[f'rmean_{w}'] = s1.rolling(w).mean().reset_index(level=0, drop=True).astype('float32')
        out[f'rstd_{w}'] = s1.rolling(w).std().reset_index(level=0, drop=True).astype('float32')
        out[f'rnz_{w}'] = s1.rolling(w).apply(lambda x: np.count_nonzero(x > 0), raw=True).reset_index(level=0, drop=True).astype('float32')

    # days since last non-zero sale (lightweight approximation)
    out['is_sale'] = (out['y'] > 0).astype('int8')
    out['sale_idx'] = out.groupby('ItemCode', observed=True, sort=False)['is_sale'].cumsum()
    out['days_since_sale'] = out.groupby(['ItemCode', 'sale_idx'], observed=True).cumcount().astype('int16')
    out.drop(columns=['is_sale', 'sale_idx'], inplace=True)

    # simple price/cost context
    gp = out.groupby('ItemCode', observed=True, sort=False)
    out['price_chg_7'] = (out['UnitPrice'] - gp['UnitPrice'].shift(7)).astype('float32')
    out['cost_chg_7'] = (out['Unit Cost'] - gp['Unit Cost'].shift(7)).astype('float32')

    return out


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = add_calendar_features(df)
    out = add_lag_rolling_features(out)
    return out


In [6]:
@dataclass
class WrmsseArtifacts:
    scale: pd.Series
    weights: pd.Series


def build_wrmsse_artifacts(train_panel: pd.DataFrame) -> WrmsseArtifacts:
    # RMSSE denominator from naive 1-step on training target
    diff = train_panel.groupby('ItemCode', observed=True)['y'].diff()
    scale = diff.pow(2).groupby(train_panel['ItemCode']).mean().replace(0, np.nan)

    # profit-based weights, clipped at 0
    profit = train_panel.groupby('ItemCode', observed=True)['profit'].sum().clip(lower=0)
    if profit.sum() <= 0:
        weights = pd.Series(1.0 / len(profit), index=profit.index)
    else:
        weights = profit / profit.sum()

    scale = scale.fillna(scale.median() if np.isfinite(scale.median()) else 1.0)
    return WrmsseArtifacts(scale=scale.astype('float64'), weights=weights.astype('float64'))


# def wrmsse_score(y_true: pd.Series, y_pred: pd.Series, sku: pd.Series, art: WrmsseArtifacts) -> float:
#     df = pd.DataFrame({'sku': sku.values, 'y_true': y_true.values, 'y_pred': y_pred.values})
#     mse_by_sku = df.groupby('sku', observed=True).apply(lambda x: np.mean((x['y_true'] - x['y_pred'])**2), include_groups=False)

#     common = mse_by_sku.index.intersection(art.scale.index).intersection(art.weights.index)
#     rmsse = np.sqrt((mse_by_sku[common] / art.scale[common]).clip(lower=0))
#     score = float((rmsse * art.weights[common]).sum())
#     return score
def wrmsse_score(y_true, y_pred, sku, art: WrmsseArtifacts) -> float:
    y_true_arr = np.asarray(y_true, dtype=np.float64)
    y_pred_arr = np.asarray(y_pred, dtype=np.float64)
    sku_arr = np.asarray(sku)

    if not (len(y_true_arr) == len(y_pred_arr) == len(sku_arr)):
        raise ValueError(
            f"Length mismatch: y_true={len(y_true_arr)}, y_pred={len(y_pred_arr)}, sku={len(sku_arr)}"
        )

    df = pd.DataFrame({
        "sku": sku_arr,
        "err2": (y_true_arr - y_pred_arr) ** 2
    })

    mse_by_sku = df.groupby("sku", observed=True)["err2"].mean()

    common = mse_by_sku.index.intersection(art.scale.index).intersection(art.weights.index)
    if len(common) == 0:
        return np.nan

    rmsse = np.sqrt((mse_by_sku.loc[common] / art.scale.loc[common]).clip(lower=0))
    return float((rmsse * art.weights.loc[common]).sum())


In [7]:
print('[4] Train/valid split (last 28 days holdout)...')
last_date = panel['Date'].max()
valid_start = last_date - pd.Timedelta(days=VAL_DAYS - 1)

feat_df = make_features(panel)
feat_df = feat_df.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

feature_cols = [
    c for c in feat_df.columns
    if c not in ['Date', 'ItemCode', 'Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'profit', 'y']
]

train_mask = feat_df['Date'] < valid_start
valid_mask = feat_df['Date'] >= valid_start

train_df = feat_df.loc[train_mask].dropna(subset=feature_cols + ['y']).copy()
valid_df = feat_df.loc[valid_mask].dropna(subset=feature_cols + ['y']).copy()

X_train, y_train = train_df[feature_cols], train_df['y']
X_valid, y_valid = valid_df[feature_cols], valid_df['y']

wr_art = build_wrmsse_artifacts(train_df[['ItemCode', 'y', 'profit']].copy())

print('train rows:', len(train_df), 'valid rows:', len(valid_df), 'features:', len(feature_cols))


[4] Train/valid split (last 28 days holdout)...
train rows: 26673240 valid rows: 447216 features: 28


In [8]:
print('[5] Train models...')
models = {}
valid_preds = {}

# LightGBM Tweedie
lgb_model = lgb.LGBMRegressor(
    objective='tweedie',
    tweedie_variance_power=1.2,
    learning_rate=0.05,
    n_estimators=700,
    num_leaves=127,
    min_child_samples=80,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)
lgb_model.fit(X_train, y_train)
models['lgb_tweedie'] = lgb_model
valid_preds['lgb_tweedie'] = np.clip(lgb_model.predict(X_valid), 0, None)

# RandomForest
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=16,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
models['rf'] = rf_model
valid_preds['rf'] = np.clip(rf_model.predict(X_valid), 0, None)

# XGBoost (optional)
if HAS_XGB:
    xgb_model = xgb.XGBRegressor(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=SEED,
        objective='reg:squarederror',
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    models['xgb'] = xgb_model
    valid_preds['xgb'] = np.clip(xgb_model.predict(X_valid), 0, None)

print('models trained:', list(models.keys()))


[5] Train models...
models trained: ['lgb_tweedie', 'rf', 'xgb']


In [9]:
print('[6] Evaluate single models and optimize ensemble weights...')

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

scores = []
for name, pred in valid_preds.items():
    r = rmse(y_valid, pred)
    w = wrmsse_score(y_valid, pred, valid_df['ItemCode'], wr_art)
    scores.append((name, r, w))

scores_df = pd.DataFrame(scores, columns=['model', 'rmse', 'wrmsse']).sort_values(['wrmsse', 'rmse'])
print(scores_df)

cand_names = list(valid_preds.keys())
P = np.column_stack([valid_preds[n] for n in cand_names])

def obj(w):
    pred = np.clip(P @ w, 0, None)
    return np.sqrt(mean_squared_error(y_valid, pred))

cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)
bounds = [(0.0, 1.0)] * len(cand_names)
x0 = np.ones(len(cand_names)) / len(cand_names)
res = minimize(obj, x0=x0, bounds=bounds, constraints=cons, method='SLSQP')

if res.success:
    w_opt = res.x
else:
    w_opt = x0

ens_pred = np.clip(P @ w_opt, 0, None)
ens_rmse = rmse(y_valid, ens_pred)
ens_wrmsse = wrmsse_score(y_valid, ens_pred, valid_df['ItemCode'], wr_art)

print('ensemble weights:', {k: float(v) for k, v in zip(cand_names, w_opt)})
print('ensemble rmse:', ens_rmse)
print('ensemble wrmsse:', ens_wrmsse)


[6] Evaluate single models and optimize ensemble weights...
         model      rmse    wrmsse
1           rf  2.422472  0.405528
0  lgb_tweedie  2.768014  0.420958
2          xgb  2.687929  0.467057
ensemble weights: {'lgb_tweedie': 0.009535150413083868, 'rf': 0.9904648495869162, 'xgb': 0.0}
ensemble rmse: 2.4224379741799917
ensemble wrmsse: 0.40524547328677574


In [10]:
print('[7] Choose final predictor...')
# Primary criterion: WRMSSE, secondary: RMSE
best_single = scores_df.iloc[0]

single_wr = float(best_single['wrmsse'])
single_rm = float(best_single['rmse'])

use_ensemble = (ens_wrmsse < single_wr) or (np.isclose(ens_wrmsse, single_wr) and ens_rmse < single_rm)

if use_ensemble:
    final_type = 'ensemble'
    final_info = {'weights': {k: float(v) for k, v in zip(cand_names, w_opt)}}
else:
    final_type = 'single'
    final_name = str(best_single['model'])
    final_info = {'model': final_name}

print('final_type:', final_type)
print('final_info:', final_info)


[7] Choose final predictor...
final_type: ensemble
final_info: {'weights': {'lgb_tweedie': 0.009535150413083868, 'rf': 0.9904648495869162, 'xgb': 0.0}}


In [11]:
print('[8] Refit on full history and forecast 56 days recursively...')
full_df = make_features(panel).sort_values(['ItemCode', 'Date']).reset_index(drop=True)
full_train = full_df.dropna(subset=feature_cols + ['y']).copy()
X_full, y_full = full_train[feature_cols], full_train['y']

refit_models = {}
for name in models.keys():
    if name == 'lgb_tweedie':
        m = lgb.LGBMRegressor(
            objective='tweedie', tweedie_variance_power=1.2,
            learning_rate=0.05, n_estimators=700, num_leaves=127,
            min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.05, reg_lambda=1.0,
            random_state=SEED, n_jobs=-1, verbosity=-1
        )
    elif name == 'rf':
        m = RandomForestRegressor(
            n_estimators=300, max_depth=16,
            min_samples_leaf=2, random_state=SEED, n_jobs=-1
        )
    elif name == 'xgb' and HAS_XGB:
        m = xgb.XGBRegressor(
            n_estimators=700, learning_rate=0.05, max_depth=10,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.05, reg_lambda=1.0,
            random_state=SEED, objective='reg:squarederror', n_jobs=-1
        )
    else:
        continue
    m.fit(X_full, y_full)
    refit_models[name] = m

work = panel[['Date', 'ItemCode', 'Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'profit', 'y']].copy()
last_date = work['Date'].max()
all_skus = work['ItemCode'].cat.categories if str(work['ItemCode'].dtype) == 'category' else work['ItemCode'].unique()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=HORIZON, freq='D')

pred_rows = []
for d in future_dates:
    step = pd.DataFrame({'Date': d, 'ItemCode': all_skus})
    step['ItemCode'] = step['ItemCode'].astype('category')
    for c in ['Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'profit']:
        step[c] = 0.0
    step['y'] = np.nan

    work = pd.concat([work, step], ignore_index=True)
    work = work.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

    feat = make_features(work)
    cur = feat[feat['Date'] == d].copy()
    X_cur = cur[feature_cols].fillna(0.0)

    if final_type == 'ensemble':
        cols = []
        names = []
        for n, m in refit_models.items():
            cols.append(np.clip(m.predict(X_cur), 0, None))
            names.append(n)
        mat = np.column_stack(cols)
        ww = np.array([final_info['weights'].get(n, 0.0) for n in names], dtype=float)
        if ww.sum() == 0:
            ww = np.ones(len(names)) / len(names)
        else:
            ww = ww / ww.sum()
        pred = np.clip(mat @ ww, 0, None)
    else:
        m = refit_models[final_info['model']]
        pred = np.clip(m.predict(X_cur), 0, None)

    work.loc[work['Date'] == d, 'y'] = pred
    pred_rows.append(pd.DataFrame({'Date': d, 'ItemCode': cur['ItemCode'].values, 'pred': pred}))

pred56 = pd.concat(pred_rows, ignore_index=True)
print('pred56 shape:', pred56.shape)
pred56.head(3)


[8] Refit on full history and forecast 56 days recursively...
pred56 shape: (894432, 3)


,Date,ItemCode,pred
0,2025-09-06,SKU-00001,6.750725e-11
1,2025-09-06,SKU-00002,2.492761e-10
2,2025-09-06,SKU-00003,2.404967e-10


In [12]:
print('[9] Build submission.csv...')
sub_template = pd.read_csv(SUB_PATH)

sku_to_pred = (
    pred56.sort_values(['ItemCode', 'Date'])
    .groupby('ItemCode', observed=True)['pred']
    .apply(list)
    .to_dict()
)

rows = []
fcols = [f'F{i}' for i in range(1, 29)]
for rid in sub_template['id']:
    sku, suffix = rid.rsplit('_', 1)
    vals = sku_to_pred.get(sku, [0.0] * 56)
    if suffix == 'validation':
        v = vals[:28]
    else:
        v = vals[28:56]
    rows.append([rid] + [float(max(0.0, x)) for x in v])

sub = pd.DataFrame(rows, columns=['id'] + fcols)

# strict checks
assert sub.shape == sub_template.shape
assert sub['id'].is_unique
assert set(sub['id']) == set(sub_template['id'])
assert np.isfinite(sub[fcols].to_numpy()).all()
assert (sub[fcols].to_numpy() >= 0).all()

sub.to_csv('submission.csv', index=False)
print('saved submission.csv with shape', sub.shape)
sub.head(3)


[9] Build submission.csv...
saved submission.csv with shape (31944, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,6.750725e-11,3.174437e-11,1.242266e-10,1.099700e-10,1.339047e-10,1.246068e-10,1.207462e-10,2.924246e-10,5.834322e-11,...,2.617145e-10,2.589165e-10,2.773014e-10,2.966287e-10,5.537817e-11,3.042922e-10,2.873841e-10,3.709290e-10,4.252902e-10,4.512395e-10
1,SKU-00002_validation,2.492761e-10,2.376552e-11,9.866193e-11,1.139430e-10,4.363468e-10,4.195121e-10,4.486407e-10,3.840023e-10,5.305253e-11,...,2.714439e-10,3.250728e-10,3.513851e-10,3.887409e-10,7.080545e-11,3.774271e-10,3.457714e-10,4.798941e-10,4.207489e-10,4.596627e-10
2,SKU-00003_validation,2.404967e-10,2.240912e-11,1.062513e-10,1.114394e-10,3.098125e-10,2.932067e-10,1.134337e-10,3.566447e-10,5.594360e-11,...,2.882455e-10,3.039937e-10,3.317220e-10,3.101206e-10,5.452072e-11,2.891457e-10,2.612412e-10,3.459785e-10,3.190232e-10,3.077125e-10
